# RideWise · Notebook 08 — Deployment & Monitoring

**Serve the churn model behind a fast API, verify latency, and set up the checks that keep it healthy in production.**

---

### What you will learn
- How to wrap a serialised model in a FastAPI scoring service
- How to verify the sub-second latency requirement
- What to monitor: performance decay and distribution drift
- A practical re-training and versioning cadence

### How to read this notebook
Every section follows the same rhythm used throughout the project:
**the business question first**, then the data, then the method, then a
**validation check** that proves the step did what we claimed. Run the cells
top to bottom; nothing depends on hidden state.

---

## 1. The business question

A model that lives in a notebook helps no one at 9am on a Monday. Operations
needs to **score a rider on demand in well under a second**. This notebook turns
the saved model into a service and defines how we keep it trustworthy over time.

In [1]:
# --- environment setup (run me first) — self-contained, no project module ---
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

RANDOM_STATE = 42
NAVY, ACCENT = "#1F3A5F", "#C8843C"
DATA_DIR  = Path("../data")  if Path("../data").exists()  else Path("data")
MODEL_DIR = Path("../models") if Path("../models").exists() else Path("models")

# The modelling feature set (note: churn_prob is the raw NOISE target and is
# deliberately excluded — using it would leak the unlearnable original label).
FEATURE_COLUMNS = [
    "recency", "frequency", "monetary", "avg_fare", "tenure",
    "avg_surge", "max_surge", "tip_rate", "trips_per_week",
    "avg_duration", "distinct_drivers", "weekend_ratio", "night_ratio",
    "card_ratio", "sessions_count", "avg_time_on_app", "avg_pages",
    "conversion_rate", "age", "avg_rating_given", "loyalty_rank",
    "was_referred",
]

# Chain-via-disk: Notebook 03 saved the analytics table; load it here.
df = pd.read_csv(DATA_DIR / "analytics_table.csv")
print(f"Loaded analytics table: {df.shape}  churn rate: {df['churn'].mean():.1%}")

Loaded analytics table: (10000, 29)  churn rate: 28.8%


In [2]:
import joblib, time, numpy as np
from pathlib import Path
mdir = MODEL_DIR
bundle = joblib.load(mdir / "churn_rf.joblib")
model, FEATS = bundle["model"], bundle["features"]
print("Loaded model expecting", len(FEATS), "features.")

Loaded model expecting 22 features.


## 2. A scoring function with a risk band

Operations does not want a bare number; they want a **band** they can act on.
We map the probability to Low / Medium / High.

In [3]:
import pandas as pd

def score_rider(features: dict) -> dict:
    row = pd.DataFrame([{f: features.get(f, 0) for f in FEATS}])
    prob = float(model.predict_proba(row)[0, 1])
    band = "High" if prob >= 0.6 else "Medium" if prob >= 0.35 else "Low"
    return {"churn_probability": round(prob, 4), "risk_band": band}

# Worked example
example = df[FEATS].iloc[0].to_dict()
print(score_rider(example))

{'churn_probability': 0.3276, 'risk_band': 'Low'}


## 3. Verify the latency requirement

In [4]:
sample = df[FEATS].iloc[:1000].to_dict(orient="records")
t0 = time.perf_counter()
for r in sample:
    score_rider(r)
elapsed = time.perf_counter() - t0
per_rider_ms = elapsed / len(sample) * 1000        # mean latency per rider, in ms
print(f"Scored {len(sample):,} riders in {elapsed*1000:.0f} ms  ->  {per_rider_ms:.2f} ms per rider")
print("Requirement: < 1 second per rider.  " + ("PASSED" if per_rider_ms < 1000 else "FAILED"))

Scored 1,000 riders in 39957 ms  ->  39.96 ms per rider
Requirement: < 1 second per rider.  PASSED


## 4. The FastAPI service

The code below is the deployable service. Save it as `app.py` in `src/` and run
`uvicorn app:app --reload`, then POST rider features to `/score`. (We show the
code here rather than launching a server inside the notebook.)

In [5]:
fastapi_code = """
from fastapi import FastAPI
from pydantic import BaseModel
import joblib, pandas as pd
from pathlib import Path

app = FastAPI(title="RideWise Churn Scoring API")
bundle = joblib.load(Path(__file__).parent.parent / "models" / "churn_rf.joblib")
model, FEATS = bundle["model"], bundle["features"]

class RiderFeatures(BaseModel):
    features: dict

@app.get("/health")
def health():
    return {"status": "ok", "n_features": len(FEATS)}

@app.post("/score")
def score(payload: RiderFeatures):
    row = pd.DataFrame([{f: payload.features.get(f, 0) for f in FEATS}])
    prob = float(model.predict_proba(row)[0, 1])
    band = "High" if prob >= 0.6 else "Medium" if prob >= 0.35 else "Low"
    return {"churn_probability": round(prob, 4), "risk_band": band}
"""
# Write it next to the pipeline so the repo is runnable
out = Path("../src"); out.mkdir(exist_ok=True)
(out / "app.py").write_text(fastapi_code)
print("Wrote FastAPI service to", out / "app.py")

Wrote FastAPI service to ..\src\app.py


## 5. Monitoring — keeping the model honest in production

A deployed model decays as the world changes. Three checks catch the common
failure modes.

**(a) Performance decay** — re-score each new labelled window and watch ROC-AUC
and PR-AUC. A sustained drop means it is time to retrain.

**(b) Distribution drift** — watch the churn base rate and the feature
distributions. If riders today look unlike the training population, predictions
are extrapolations.

**(c) Segment stability** — the K-means segment shares should be roughly stable
month to month; a sudden shift signals a behavioural change worth investigating.

In [6]:
# Simple drift check: compare a feature's distribution in two halves of the data
import numpy as np
from scipy.stats import ks_2samp
ref = df["recency"].sample(2000, random_state=1)
new = df["recency"].sample(2000, random_state=2)
stat, pval = ks_2samp(ref, new)
print(f"KS drift test on 'recency': statistic={stat:.3f}, p={pval:.3f}")
print("p > 0.05  ->  no significant drift (as expected for one static dataset).")

KS drift test on 'recency': statistic=0.021, p=0.770
p > 0.05  ->  no significant drift (as expected for one static dataset).


## 6. A practical cadence

- **Daily:** log every scoring request with latency; alert on API errors or slow responses.
- **Weekly:** refresh the retention list from the latest snapshot.
- **Monthly:** recompute ROC-AUC/PR-AUC on freshly labelled riders; check drift and segment shares.
- **Quarterly (or on decay):** retrain, version the model, and archive metrics for comparison.

## 7. Summary — the full loop is closed

From five raw tables we built a clean analytics table, segmented riders,
trained and explained churn models, turned scores into a targeted retention
list, and served the model behind a fast API with a monitoring plan. Each step
answered a specific business challenge with a specific method — exactly the
mapping laid out in the work plan.